# Definitions

In [3]:
import torch
import torch.nn as nn
import torchaudio
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import os
from typing import List
import torch.nn.functional as F
import torch.optim as optim

In [13]:
AUDIO_DIR = "../../data_processing/separate_scripts/golden_audio"
LABEL_DIR = "../../data_processing/separate_scripts/golden_breaks"
INDEX_SET = [0, 6, 19]  # 16
interval_width = 20  # ms
NUM_CLASSES = 2

In [14]:
def create_soft_labels(labels: torch.Tensor, sigma: float = 2.0) -> torch.Tensor:
    """
    Create soft labels by applying a Gaussian decay around positive labels.
    
    Args:
        labels: Binary tensor of shape (sequence_length,)
        sigma: Standard deviation for Gaussian decay (controls spread)
    
    Returns:
        Soft labels tensor of shape (sequence_length,)
    """
    soft_labels = labels.float().clone()
    positive_indices = torch.where(labels == 1)[0]

    # Create position tensor for vectorized distance calculation
    positions = torch.arange(len(labels), device=labels.device)

    # For each positive label, apply Gaussian decay to nearby positions
    for pos_idx in positive_indices:
        # Calculate distances to current positive index
        distances = torch.abs(positions - pos_idx)
        # Apply Gaussian decay
        decay = torch.exp(-(distances ** 2) / (2 * sigma ** 2))
        # Update soft labels, taking maximum with existing values
        soft_labels = torch.maximum(soft_labels, decay)

    return soft_labels

In [15]:
class BreakDataset(Dataset):
    def __init__(self, indices: List[int] = INDEX_SET, audio_dir: str = AUDIO_DIR, label_dir: str = LABEL_DIR, sigma: float = 2.0):
        self.indices = indices
        self.audio_dir = audio_dir
        self.label_dir = label_dir

        self.data = []
        for idx in indices:
            # Load audio and convert to spectrogram
            audio_path = os.path.join(audio_dir, f"{idx}.mp3")
            waveform, sr = torchaudio.load(audio_path)

            # Ensure mono channel
            waveform = waveform.mean(dim=0, keepdim=True)

            # Compute mel spectrogram
            mel_spec = torchaudio.transforms.MelSpectrogram(
                sample_rate=sr,
                hop_length=int(sr * interval_width / 1000),
                n_mels=32
            )(waveform)

            mel_spec = mel_spec.squeeze(0)

            # Add normalization
            mel_spec = torch.log(mel_spec + 1e-9)  # Log scale
            mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-9)  # Normalize

            # Load labels
            label_path = os.path.join(label_dir, f"{idx}.csv")
            df = pd.read_csv(label_path)
            hard_labels = torch.tensor(df['break'].values, dtype=torch.bool)

            # Create soft labels
            soft_labels = create_soft_labels(hard_labels, sigma)

            # Truncate to last break + 2
            last_break = hard_labels.nonzero(as_tuple=True)[0][-1]
            mel_spec = mel_spec[:, :last_break + 2]
            soft_labels = soft_labels[:last_break + 2]
            hard_labels = hard_labels[:last_break + 2]

            self.data.append((mel_spec, soft_labels, hard_labels))


    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        return self.data[idx]

In [16]:
class CNNRNN(nn.Module):
    def __init__(self, n_mels=32, hidden_size=64, num_layers=3):
        super(CNNRNN, self).__init__()
        # self.conv = nn.Sequential(
        #     nn.Conv1d(n_mels, 64, kernel_size=3, padding=1),
        #     nn.ReLU(),
        #     nn.Conv1d(64, 32, kernel_size=3, padding=1),
        #     nn.ReLU()
        # )
        # self.rnn = nn.LSTM(
        #     input_size=32,
        #     hidden_size=hidden_size,
        #     num_layers=num_layers,
        #     batch_first=True,
        #     bidirectional=True
        # )

        self.conv = nn.Sequential(
            nn.Conv1d(n_mels, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU()
        )

        self.rnn = nn.LSTM(
            input_size=32,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.2 if num_layers > 1 else 0
        )
        
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, NUM_CLASSES)
        )

    def forward(self, x, mask):
        x = self.conv(x)
        x = x.permute(0, 2, 1)
        x, _ = self.rnn(x)
        x = self.fc(x)
        x = x * mask.unsqueeze(-1)
        return x

# Training

In [17]:
MODEL_PATH = 'best_model_soft_labels.pth'

In [18]:
# Split data into train and validation sets
total_indices = len(INDEX_SET)
train_size = int(0.75 * total_indices)
train_indices = INDEX_SET[:train_size]
val_indices = INDEX_SET[train_size:]

train_dataset = BreakDataset(train_indices, AUDIO_DIR, LABEL_DIR)
val_dataset = BreakDataset(val_indices, AUDIO_DIR, LABEL_DIR)

train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=1, shuffle=False)

# Initialize the model, optimizer, and loss function
model = CNNRNN()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2)

# Calculate class weights for the loss function
all_labels = torch.cat([labels for _, labels, _ in train_dataset])
pos_weight = torch.tensor([(all_labels == 0).sum() / (all_labels == 1).sum()])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Training loop with validation and early stopping
epochs = 30  # Increased epochs since we have early stopping
best_loss = float('inf')
patience = 10
patience_counter = 0

for epoch in range(epochs):
    # Training phase
    model.train()
    train_loss = 0
    train_batches = 0

    for batch in train_dataloader:
        mel_spec, soft_labels, hard_labels = batch
        max_length = mel_spec.shape[2]
        mel_specs = []
        soft_labels_list = []

        for i in range(mel_spec.shape[0]):
            mel_specs.append(F.pad(mel_spec[i], (0, max_length - mel_spec[i].shape[1])))
            soft_labels_list.append(F.pad(soft_labels[i], (0, max_length - soft_labels[i].shape[0])))

        mel_specs = torch.stack(mel_specs)
        soft_labels = torch.stack(soft_labels_list)
        mask = (mel_specs != 0).any(dim=1).float()

        optimizer.zero_grad()
        output = model(mel_specs, mask)

        # Use MSE loss for soft labels
        loss = F.mse_loss(
            torch.sigmoid(output.view(-1, 2)[:, 1]),
            soft_labels.view(-1)
        )

        loss.backward()
        optimizer.step()


        train_loss += loss.item()
        train_batches += 1

    avg_train_loss = train_loss / train_batches

    # Validation phase
    model.eval()
    val_loss = 0
    val_batches = 0

    with torch.no_grad():
        for batch in val_dataloader:
            mel_spec, labels, _ = batch
            max_length = mel_spec.shape[2]
            mel_specs = []
            labels_list = []

            for i in range(mel_spec.shape[0]):
                mel_specs.append(F.pad(mel_spec[i], (0, max_length - mel_spec[i].shape[1])))
                labels_list.append(F.pad(labels[i], (0, max_length - labels[i].shape[0])))

            mel_specs = torch.stack(mel_specs)
            labels = torch.stack(labels_list)
            mask = (mel_specs != 0).any(dim=1).float()

            output = model(mel_specs, mask)
            loss = criterion(output.view(-1, 2)[:, 1], labels.view(-1).float())

            val_loss += loss.item()
            val_batches += 1

    avg_val_loss = val_loss / val_batches

    # Learning rate scheduling
    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch + 1}")
    print(f"Training Loss: {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")

    # Early stopping check
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print("Saved new best model!")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered!")
            break

    print("-" * 50)

print("Training completed!")


Epoch 1
Training Loss: 0.1764
Validation Loss: 1.3214
Learning Rate: 0.001000
Saved new best model!
--------------------------------------------------
Epoch 2
Training Loss: 0.1694
Validation Loss: 1.3407
Learning Rate: 0.001000
--------------------------------------------------
Epoch 3
Training Loss: 0.1631
Validation Loss: 1.3697
Learning Rate: 0.001000
--------------------------------------------------
Epoch 4
Training Loss: 0.1539
Validation Loss: 1.4166
Learning Rate: 0.000100
--------------------------------------------------
Epoch 5
Training Loss: 0.1452
Validation Loss: 1.4240
Learning Rate: 0.000100
--------------------------------------------------
Epoch 6
Training Loss: 0.1438
Validation Loss: 1.4331
Learning Rate: 0.000100
--------------------------------------------------
Epoch 7
Training Loss: 0.1421
Validation Loss: 1.4443
Learning Rate: 0.000010
--------------------------------------------------
Epoch 8
Training Loss: 0.1410
Validation Loss: 1.4468
Learning Rate: 0.0000

# Inference

In [19]:
import torch
import torchaudio
import pandas as pd
import os
import torch.nn.functional as F


# # Reuse the model definition from training script
# class CNNRNN(nn.Module):
#     def __init__(self, n_mels=32, hidden_size=128, num_layers=2):
#         super(CNNRNN, self).__init__()
#         self.conv = nn.Sequential(
#             nn.Conv1d(n_mels, 64, kernel_size=3, padding=1),
#             nn.ReLU(),
#             nn.Conv1d(64, 32, kernel_size=3, padding=1),
#             nn.ReLU()
#         )
#         self.rnn = nn.LSTM(
#             input_size=32,
#             hidden_size=hidden_size,
#             num_layers=num_layers,
#             batch_first=True,
#             bidirectional=True
#         )
#         self.fc = nn.Sequential(
#             nn.Linear(hidden_size * 2, 64),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(64, NUM_CLASSES)
#         )
# 
#     def forward(self, x, mask):
#         x = self.conv(x)
#         x = x.permute(0, 2, 1)
#         x, _ = self.rnn(x)
#         x = self.fc(x)
#         x = x * mask.unsqueeze(-1)
#         return x

def prepare_input(audio_path, interval_width=20):
    # Load audio
    waveform, sr = torchaudio.load(audio_path)

    # Ensure mono channel
    waveform = waveform.mean(dim=0, keepdim=True)

    # Compute mel spectrogram
    mel_spec_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=sr,
        hop_length=int(sr * interval_width / 1000),
        n_mels=32
    )
    mel_spec = mel_spec_transform(waveform)
    mel_spec = mel_spec.squeeze(0)

    # Normalization (same as in training)
    mel_spec = torch.log(mel_spec + 1e-9)
    mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-9)

    return mel_spec, sr

def inference(model_path, audio_path, output_csv_path):
    # Load the trained model
    model = CNNRNN()
    model.load_state_dict(torch.load(model_path, weights_only=True))
    model.eval()

    # Prepare input
    mel_spec, sr = prepare_input(audio_path)

    # Prepare batch dimensions
    mel_spec = mel_spec.unsqueeze(0)  # Add batch dimension

    # Create mask
    mask = (mel_spec != 0).any(dim=1).float()

    # Inference
    with torch.no_grad():
        outputs = model(mel_spec, mask)

    # Convert outputs to probabilities
    probabilities = torch.sigmoid(outputs.squeeze())

    # Convert to numpy for easier handling
    probs_np = probabilities.numpy()

    # Create DataFrame
    df = pd.DataFrame({
        'time_ms': [i * interval_width for i in range(len(probs_np))],
        'break_probability': probs_np[:, 1],  # Probability of being a break
        'is_break': (probs_np[:, 1] > 0.5).astype(int)  # Threshold at 0.5
    })

    # Save to CSV
    df.to_csv(output_csv_path, index=False)

    print(f"Inference completed. Results saved to {output_csv_path}")
    return df

# # Example usage
# if __name__ == "__main__":
#     # Paths
#     MODEL_PATH = 'best_model.pth'
#     AUDIO_PATH = '../../data_processing/separate_scripts/golden_audio/16.mp3'
#     OUTPUT_CSV_PATH = 'break_predictions.csv'
# 
#     # Run inference
#     results = inference(MODEL_PATH, AUDIO_PATH, OUTPUT_CSV_PATH)
# 
#     # Optional: print some statistics
#     print(f"Total predictions: {len(results)}")
#     print(f"Number of predicted breaks: {results['is_break'].sum()}")


In [20]:
# for an entire directory of audio (all following the name format of {INTEGER}.mp3), run inference on all of them and put their corresponding csvs in a directory
input_dir = '../../data/clean/audio/vocals'
output_dir = 'predictions'
model_dir = 'best_model_soft_labels.pth'

def inference_dir(model_path, input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for audio_file in os.listdir(input_dir):
        if not audio_file.endswith('.mp3'):
            continue
        audio_path = os.path.join(input_dir, audio_file)
        output_csv_path = os.path.join(output_dir, f"{audio_file[:-4]}_predictions.csv")
        inference(model_path, audio_path, output_csv_path)
        
# Run inference on the directory
inference_dir(model_dir, input_dir, output_dir)

Inference completed. Results saved to predictions/0_predictions.csv
Inference completed. Results saved to predictions/10_predictions.csv
Inference completed. Results saved to predictions/11_predictions.csv
Inference completed. Results saved to predictions/12_predictions.csv
Inference completed. Results saved to predictions/13_predictions.csv
Inference completed. Results saved to predictions/14_predictions.csv
Inference completed. Results saved to predictions/15_predictions.csv
Inference completed. Results saved to predictions/16_predictions.csv
Inference completed. Results saved to predictions/17_predictions.csv
Inference completed. Results saved to predictions/18_predictions.csv
Inference completed. Results saved to predictions/19_predictions.csv
Inference completed. Results saved to predictions/1_predictions.csv
Inference completed. Results saved to predictions/20_predictions.csv
Inference completed. Results saved to predictions/21_predictions.csv
Inference completed. Results saved t

In [21]:
import pandas as pd
import os

# Get all prediction files
pred_dir = 'predictions'
break_dir = 'breaks'
os.makedirs(break_dir, exist_ok=True)

for filename in os.listdir(pred_dir):
    if filename.endswith('_predictions.csv'):
        # Read predictions file
        pred_df = pd.read_csv(os.path.join(pred_dir, filename))

        # Create breaks dataframe
        breaks_df = pd.DataFrame({
            'start': pred_df['time_ms'],
            'end': pred_df['time_ms'] + 20,
            'break': pred_df['is_break']
        })

        # Get the integer prefix from filename
        prefix = filename.split('_')[0]

        # Save to breaks directory
        output_filename = f'{prefix}_breaks.csv'
        breaks_df.to_csv(os.path.join(break_dir, output_filename), index=False)


In [22]:
import pandas as pd
import os

break_dir = 'breaks'
all_breaks = []

for filename in sorted(os.listdir(break_dir)):
    if filename.endswith('_breaks.csv'):
        # Read break file
        df = pd.read_csv(os.path.join(break_dir, filename))

        # Extract file number
        file_num = int(filename.split('_')[0])

        # Add file column
        df['file'] = file_num

        all_breaks.append(df)

# Concatenate all dataframes
if all_breaks:
    final_df = pd.concat(all_breaks, ignore_index=True)

    # Reorder columns and add index
    final_df = final_df.reset_index().rename(columns={'index': 'index'})
    final_df = final_df[['index', 'file', 'start', 'end', 'break']]
    
    # remove last row if the last row itself contains any null values
    if final_df.iloc[-1].isnull().values.any():
        final_df = final_df[:-1]
    
    # collapse rows between breaks = True and remove break column
    # example 1: index, file, start, end, break
    #                0,    0,     0,  20, True
    #                1,    0,    20,  40, False
    #                2,    0,    40,  60, False
    #                3,    0,    60,  80, True
    #                4,    0,    80, 100, True
    #
    # becomes:   index, file, start, end,
    #                0,    0,     0,  80,
    #                1,    0,    80, 100, 
    #
    
    # example 2: index, file, start, end, break
    #                0,    0,     0,  20, True
    #                1,    0,    20,  40, False
    #                2,    0,    40,  60, False
    #                3,    4,    60,  80, True
    
    # becomes:   index, file, start, end,
    #                0,    0,     0,  60,
    #                1,    4,    60,  80,
    # since (1) we don't collapse rows of different files, and
    #       (2) the last row for a file is treated as a break
    
    # example 3: index, file, start, end, break
    #                0,    0,     0,  20, True
    #                1,    0,    20,  40, False
    #                2,    1,     0,  20, False
    #                3,    1,    20,  40, True
    
    # becomes:   index, file, start, end,
    #                0,    0,     0,  40,
    #                1,    1,     0,  40,
    
    # [ INSERT CODE HERE ]

    # drop index column
    final_df.drop(columns=['index'], inplace=True)
    
    # iterate through all the rows
    collapsed_rows = []
    first_of_new_chain = True
    last_file = final_df.iloc[0]['file'].copy()
    current_start = None
    current_end = None
    
    for index, row in final_df.iterrows():
        if row['file'] != last_file:
            first_of_new_chain = True
            collapsed_rows.append({
                'file': last_file,
                'start': current_start,
                'end': current_end,
            })
        if first_of_new_chain:
            current_start = row['start']
            first_of_new_chain = False
        if row['break']:
            collapsed_rows.append({
                'file': row['file'],
                'start': current_start,
                'end': row['end']
            })
            first_of_new_chain = True
        last_file = row['file']
        current_end = row['end']
    
    # Save to csv
    final_df.to_csv('uncollapsed_segment_breaks.csv', index=False)
    
    
    # Create new dataframe from collapsed rows
    final_df = pd.DataFrame(collapsed_rows)


    # Save to csv
    final_df.to_csv('segment_breaks.csv', index=False)
